# EDA first pass: target pollutant and discretization (Issue #4)

Goal: pick the target pollutant and propose bins for the target, sensor and weather variables.

Sections
1. Load and sanity check
2. Missingness per column
3. Gap structure of the ground-truth (GT) columns
4. Sensor vs. GT agreement and drift over the year
5. Histograms of candidate targets
6. Train/test split and tertile bins
7. Comparison config: health-band bins for the target
8. Key numbers for the proposal

In [ ]:
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
%load_ext autoreload
%autoreload 2

from cse250a.modules.config import config

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load and sanity check

In [ ]:
df = pd.read_csv(config.paths.air_quality_csv, sep=";", decimal=",")
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]   # two empty trailing cols
df = df.dropna(how="all")                                # blank trailing rows
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H.%M.%S")
df = df.drop(columns=["Date", "Time"]).set_index("datetime")
df = df.replace(-200, np.nan)                            # missing-value tag

GT = ["CO(GT)", "NMHC(GT)", "C6H6(GT)", "NOx(GT)", "NO2(GT)"]
SENSORS = ["PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)", "PT08.S5(O3)"]
WEATHER = ["T", "RH", "AH"]

print(df.shape)
print("range:", df.index.min(), "->", df.index.max())
print("all numeric:", all(pd.api.types.is_numeric_dtype(df[c]) for c in df))
print("strictly hourly:", (df.index.to_series().diff().dropna() == pd.Timedelta("1h")).all())

**Note:** the file has **9,357** rows (not 9,358 — the UCI page counts differently) and runs to
**4 Apr 2005**, not Feb 2005. The issue text and proposal should use these values.

## 2. Missingness per column

In [ ]:
miss = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
ax = miss.plot.barh(figsize=(6, 4), color="#4C72B0")
ax.set_xlabel("% hours missing"); ax.set_title("Missingness per column")
for i, v in enumerate(miss):
    ax.text(v + 0.5, i, f"{v}%", va="center", fontsize=8)
plt.tight_layout(); plt.savefig(config.paths.figures_dir / "muska" / "missingness.png"); plt.show()
miss

In [ ]:
# Do sensors and weather go missing together (device outage)?
dev_out = df[SENSORS + WEATHER].isna()
print("rows where all sensors+weather missing:", dev_out.all(axis=1).sum())
print("rows where some but not all missing:   ", (dev_out.any(axis=1) & ~dev_out.all(axis=1)).sum())

# NMHC(GT): which months is it observed in?
df["NMHC(GT)"].notna().groupby(df.index.to_period("M")).mean().round(2)

## 3. Gap structure of the GT columns

In [ ]:
def gap_lengths(s: pd.Series) -> pd.Series:
    """Lengths (hours) of each run of consecutive NaNs."""
    m = s.isna().astype(int)
    run_id = (m != m.shift()).cumsum()
    runs = m.groupby(run_id).agg(["first", "size"])
    return runs.loc[runs["first"] == 1, "size"]

rows = []
for c in GT + ["PT08.S1(CO)"]:
    g = gap_lengths(df[c])
    rows.append({
        "column": c, "% missing": round(df[c].isna().mean() * 100, 1),
        "n_gaps": len(g), "single-hour gaps": int((g == 1).sum()),
        "longest gap (h)": int(g.max()), "% missing hrs in gaps >24h": round(g[g > 24].sum() / g.sum() * 100, 1),
    })
gaps = pd.DataFrame(rows).set_index("column")
gaps

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))
show = ["CO(GT)", "NOx(GT)", "NO2(GT)", "C6H6(GT)", "NMHC(GT)", "PT08.S1(CO)"]
for i, c in enumerate(show):
    miss_t = df.index[df[c].isna()]
    ax.scatter(miss_t, np.full(len(miss_t), i), marker="|", s=40, color="#C44E52", linewidths=0.6)
ax.set_yticks(range(len(show)), show); ax.set_title("Missing hours over time (each tick = one missing hour)")
plt.tight_layout(); plt.savefig(config.paths.figures_dir / "muska" / "missing_timeline.png"); plt.show()

In [ ]:
# Is missingness related to time of day? (relevant to the MAR assumption)
by_hour = df[["CO(GT)", "NO2(GT)"]].isna().groupby(df.index.hour).mean() * 100
ax = by_hour.plot(figsize=(6, 3), marker="o"); ax.set_xlabel("hour of day"); ax.set_ylabel("% missing")
ax.set_title("GT missingness by hour of day"); plt.tight_layout(); plt.savefig(config.paths.figures_dir / "muska" / "missing_by_hour.png"); plt.show()
by_hour.round(1).T

**Interpretation:** CO/NOx/NO2 GT have many single-hour gaps (scattered), but most *missing hours* sit in a few
long outages (up to 173 h). NMHC(GT) only exists in the first ~2 months → unusable as a target.
The by-hour plot shows the short gaps are **not** random in time: ~54% of CO(GT) is missing at 04:00 and
~93% of NO2(GT) at 03:00 (vs ~15% at other hours), which looks like a daily analyzer calibration slot.
So missingness depends on hour of day → plain MCAR is wrong, but **MAR given Hour** is plausible, and the
Hour node in Phase 1 covers it. The long outages are tied to calendar periods (see timeline); EM results
should be checked by season.

## 4. Sensor vs. GT agreement and drift

In [ ]:
pairs = [("CO(GT)", "PT08.S1(CO)"), ("C6H6(GT)", "PT08.S2(NMHC)"),
         ("NOx(GT)", "PT08.S3(NOx)"), ("NO2(GT)", "PT08.S4(NO2)")]
corr = df[[g for g, _ in pairs] + SENSORS + WEATHER].corr("spearman").loc[[g for g, _ in pairs], SENSORS + WEATHER]
corr.round(2)

In [ ]:
# Benzene GT vs its sensor: is it really an independent reference?
x = df[["C6H6(GT)", "PT08.S2(NMHC)"]].dropna()
p = np.polyfit(x["PT08.S2(NMHC)"], x["C6H6(GT)"], 3)
r2 = 1 - np.var(x["C6H6(GT)"] - np.polyval(p, x["PT08.S2(NMHC)"])) / np.var(x["C6H6(GT)"])
print(f"C6H6(GT) ~ cubic(PT08.S2): R^2 = {r2:.5f}")
print("C6H6(GT) missing exactly when S2 missing:", (df["C6H6(GT)"].isna() == df["PT08.S2(NMHC)"].isna()).all())

**Interpretation:** R² ≈ 1.0000 and identical missingness: C6H6(GT) behaves like a deterministic transform of the S2 sensor,
not an independent analyzer reading. Modelling it as "ground truth" would be circular → **exclude benzene as target.**

In [ ]:
# Drift: fit GT ~ quadratic(sensor) on the first 2 months, then look at monthly median residual.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
drift = {}
for ax, (g, s) in zip(axes, [("CO(GT)", "PT08.S1(CO)"), ("NOx(GT)", "PT08.S3(NOx)"), ("NO2(GT)", "PT08.S4(NO2)")]):
    d = df[[g, s]].dropna()
    ref = d[:"2004-04"]
    coef = np.polyfit(ref[s], ref[g], 2)
    resid = d[g] - np.polyval(coef, d[s])
    m = resid.groupby(d.index.to_period("M")).median()
    rel = m / d[g].median() * 100
    drift[g] = rel.round(0)
    ax.axhline(0, color="grey", lw=0.8); ax.plot(rel.index.to_timestamp(), rel.values, marker="o")
    ax.set_title(f"{g} vs {s}"); ax.set_ylabel("median residual, % of GT median")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Sensor→GT calibration drift (calibrated on Mar–Apr 2004)")
plt.tight_layout(); plt.savefig(config.paths.figures_dir / "muska" / "drift.png"); plt.show()
pd.DataFrame(drift).T

In [ ]:
# Same story, raw: monthly GT vs sensor scatter coloured by month
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (g, s) in zip(axes, [("CO(GT)", "PT08.S1(CO)"), ("NO2(GT)", "PT08.S4(NO2)")]):
    d = df[[g, s]].dropna()
    sc = ax.scatter(d[s], d[g], c=(d.index - d.index[0]).days, s=2, cmap="viridis", alpha=0.5)
    ax.set_xlabel(s); ax.set_ylabel(g); ax.set_title(f"{g} vs {s}")
fig.colorbar(sc, ax=axes, label="days since start")
plt.savefig(config.paths.figures_dir / "muska" / "sensor_vs_gt.png", bbox_inches="tight"); plt.show()

**Interpretation:** CO tracks its sensors well (ρ ≈ 0.88 with its nominal S1, 0.93 with S2) and its drift is
modest and not trending (monthly residual −8% to +33% of the median). NO2 barely tracks its *nominal* sensor
(ρ = 0.15 with S4) and is better explained by S3/S5 — strong cross-sensitivity, useful for Phase 2 but bad for
Phase 1. Its calibration also **drifts from November on**: Jan–Apr 2005 the reference runs 72–116% above what
the spring calibration predicts. A time-based test split for NO2 would mostly measure drift, not the model.

## 5. Histograms of candidate targets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, c in zip(axes, ["CO(GT)", "NO2(GT)", "NOx(GT)"]):
    v = df[c].dropna()
    ax.hist(v, bins=60, color="#4C72B0")
    for q in v.quantile([1/3, 2/3]):
        ax.axvline(q, color="#C44E52", ls="--")
    ax.set_title(f"{c}  (skew={v.skew():.2f})")
fig.suptitle("Candidate targets (red = tertile edges, full data)")
plt.tight_layout(); plt.savefig(config.paths.figures_dir / "muska" / "target_hist.png"); plt.show()

## 6. Train/test split and tertile bins

Proposed split (time-based, shared across all phases): **train = 10 Mar – 31 Dec 2004, test = 1 Jan – 4 Apr 2005**
(~76% / 24%). Bins are computed on train only.

In [ ]:
SPLIT = "2004-12-31 23:00"
train, test = df.loc[:SPLIT], df.loc[SPLIT:].iloc[1:]
print(f"train {len(train)} rows ({len(train)/len(df):.0%}), test {len(test)} rows")

TARGET = "CO(GT)"
BIN_VARS = [TARGET] + SENSORS + WEATHER
edges = {c: train[c].quantile([1/3, 2/3]).round(3).tolist() for c in BIN_VARS}

def occupancy(s, e):
    b = pd.cut(s.dropna(), [-np.inf, *e, np.inf], labels=["low", "med", "high"])
    return (b.value_counts(normalize=True).reindex(["low", "med", "high"]) * 100).round(0)

tab = pd.DataFrame({c: {"edge 1/3": edges[c][0], "edge 2/3": edges[c][1],
                        **{f"test % {k}": v for k, v in occupancy(test[c], edges[c]).items()}}
                    for c in BIN_VARS}).T
tab

**Interpretation:** Test-set occupancy shows how far each variable shifts from train to test. Train is 33/33/33 by construction;
large deviations in test are expected and are a limitation of a time-based split, not a bug. The CO target
itself shifts only mildly (41/30/29). Weather does shift strongly: 88% of test hours fall in the "low" T tertile
and 86% in "low" AH, and PT08.S4 also collapses to "low" (84%). The W→Sensor CPT rows for cold hours are learned
mostly from Mar and Nov/Dec. We should report this and consider a seasonal/blocked split as a robustness check.

In [ ]:
# Sparsity check for Phase 1: Ref -> S_i <- W.  With 3 bins: P(S_i | Ref, T, RH) has 3*3*3 = 27 parent configs.
for k in [3, 5, 7]:
    q = np.linspace(0, 1, k + 1)[1:-1]
    b = pd.DataFrame({c: pd.cut(train[c], [-np.inf, *train[c].quantile(q), np.inf], labels=False) for c in [TARGET, "T", "RH"]}).dropna()
    counts = b.groupby([TARGET, "T", "RH"]).size().reindex(list(product(range(k), repeat=3)), fill_value=0)
    print(f"{k} bins: {k**3:4d} parent configs | min count {counts.min():4d} | configs with <20 rows: {(counts < 20).sum()}")

## 7. Comparison config: health-band bins for the target

In [ ]:
# CO: US EPA AQI breakpoints are in ppm (8-h). 1 ppm CO ~= 1.145 mg/m^3 at 25 C, 1 atm.
co_epa_mg = [4.4 * 1.145, 9.4 * 1.145]        # ~5.0, ~10.8 mg/m^3
co = df["CO(GT)"].dropna()
print("CO, EPA bands (mg/m^3):", [round(e, 1) for e in co_epa_mg])
print((pd.cut(co, [-np.inf, *co_epa_mg, np.inf], labels=["Good", "Moderate", "USG+"]).value_counts(normalize=True) * 100).round(2))

# NO2: EEA EAQI (2024 revision) hourly bands, ug/m^3  -- verify against the EEA report before using
eaqi = [10, 25, 60, 100, 150]
no2 = df["NO2(GT)"].dropna()
print("\nNO2, EEA EAQI bands:")
print((pd.cut(no2, [-np.inf, *eaqi, np.inf], labels=["Good", "Fair", "Moderate", "Poor", "Very poor", "Extr. poor"])
       .value_counts(normalize=True).sort_index() * 100).round(1))

**Interpretation:** CO health bands put ~95% of hours in "Good" → outer CPT rows would be learned from a handful of
hours and the target is nearly constant. NO2's EAQI bands are better balanced but "Good"/"Fair" are almost empty
at a roadside site. Health bands are therefore a *comparison* configuration only, as the issue suggests.

## 8. Key numbers for the proposal

In [ ]:
co_gaps = gap_lengths(df["CO(GT)"])
summary = {
    "rows": len(df),
    "date range": f"{df.index.min():%d %b %Y} – {df.index.max():%d %b %Y}",
    "% CO(GT) missing": round(df["CO(GT)"].isna().mean() * 100, 1),
    "longest CO(GT) gap (h)": int(co_gaps.max()),
    "% sensor/weather missing (device outage)": round(df["PT08.S1(CO)"].isna().mean() * 100, 1),
    "% NMHC(GT) missing": round(df["NMHC(GT)"].isna().mean() * 100, 1),
    "train/test rows": f"{len(train)} / {len(test)}",
}
pd.Series(summary)